# 환경설정

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import koreanize_matplotlib
import seaborn as sns
from sqlalchemy import create_engine

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

In [ ]:
from dotenv import load_dotenv
load_dotenv()

engine = create_engine(os.environ["DB_URL"])
conn = engine.connect()

# 전처리

In [ ]:
DATE_FMT = "%%%%Y-%%%%m-%%%%d %%%%H:%%%%i:%%%%s.%%%%f"

RETENTION_START_HOUR = 24
RETENTION_WINDOW_DAY = 7

In [ ]:
def _normalize_sql(sql: str) -> str:

    return sql.replace("%%", "%")


def run_query(query: str, name: str | None = None) -> pd.DataFrame:
    result = conn.exec_driver_sql(_normalize_sql(query))
    rows = result.fetchall()
    df = pd.DataFrame(rows, columns=result.keys())
    if name:
        print(f"[{name}] rows={len(df):,}, cols={len(df.columns):,}")
    return df


def execute_many(sql: str) -> None:
    statements = [stmt.strip() for stmt in sql.split(";") if stmt.strip()]
    for stmt in statements:
        conn.exec_driver_sql(_normalize_sql(stmt))
    try:
        conn.commit()
    except Exception:
        pass
    print(f"Executed {len(statements):,} statements.")

## 데이터 전처리

### VIEW 생성 및 결측값 확인

- 이 과정에서 user_id 결측 + event_time 변환 실패 행 제거

In [ ]:
create_views_sql = """
-- 1) v_events_signup
DROP VIEW IF EXISTS v_events_signup;
CREATE VIEW v_events_signup AS
SELECT
    user_id,
    STR_TO_DATE(event_ts, '%%%%Y-%%%%m-%%%%d %%%%H:%%%%i:%%%%s.%%%%f') AS event_time
FROM events_signup
WHERE user_id IS NOT NULL
  AND user_id <> ''
  AND STR_TO_DATE(event_ts, '%%%%Y-%%%%m-%%%%d %%%%H:%%%%i:%%%%s.%%%%f') IS NOT NULL;

-- 2) v_events_content_start
DROP VIEW IF EXISTS v_events_content_start;
CREATE VIEW v_events_content_start AS
SELECT
    user_id,
    STR_TO_DATE(event_ts, '%%%%Y-%%%%m-%%%%d %%%%H:%%%%i:%%%%s.%%%%f') AS event_time,
    `content_id` AS content_id
FROM events_content_start
WHERE user_id IS NOT NULL
  AND user_id <> ''
  AND STR_TO_DATE(event_ts, '%%%%Y-%%%%m-%%%%d %%%%H:%%%%i:%%%%s.%%%%f') IS NOT NULL;

-- 3) v_events_lesson_view
DROP VIEW IF EXISTS v_events_lesson_view;
CREATE VIEW v_events_lesson_view AS
SELECT
    user_id,
    STR_TO_DATE(event_ts, '%%%%Y-%%%%m-%%%%d %%%%H:%%%%i:%%%%s.%%%%f') AS event_time,
    `content_id` AS content_id,
    `lesson_id`  AS lesson_id
FROM events_lesson_view
WHERE user_id IS NOT NULL
  AND user_id <> ''
  AND STR_TO_DATE(event_ts, '%%%%Y-%%%%m-%%%%d %%%%H:%%%%i:%%%%s.%%%%f') IS NOT NULL;

-- 4) v_events_lesson_complete
DROP VIEW IF EXISTS v_events_lesson_complete;
CREATE VIEW v_events_lesson_complete AS
SELECT
    user_id,
    STR_TO_DATE(event_ts, '%%%%Y-%%%%m-%%%%d %%%%H:%%%%i:%%%%s.%%%%f') AS event_time,
    `content_id` AS content_id,
    `lesson_id`  AS lesson_id
FROM events_lesson_complete
WHERE user_id IS NOT NULL
  AND user_id <> ''
  AND STR_TO_DATE(event_ts, '%%%%Y-%%%%m-%%%%d %%%%H:%%%%i:%%%%s.%%%%f') IS NOT NULL;

-- 5) v_events_content_end
DROP VIEW IF EXISTS v_events_content_end;
CREATE VIEW v_events_content_end AS
SELECT
    user_id,
    STR_TO_DATE(event_ts, '%%%%Y-%%%%m-%%%%d %%%%H:%%%%i:%%%%s.%%%%f') AS event_time,
    `content_id` AS content_id
FROM events_content_end
WHERE user_id IS NOT NULL
  AND user_id <> ''
  AND STR_TO_DATE(event_ts, '%%%%Y-%%%%m-%%%%d %%%%H:%%%%i:%%%%s.%%%%f') IS NOT NULL;

-- 6) v_events_related_question_click
DROP VIEW IF EXISTS v_events_related_question_click;
CREATE VIEW v_events_related_question_click AS
SELECT
    user_id,
    STR_TO_DATE(event_ts, '%%%%Y-%%%%m-%%%%d %%%%H:%%%%i:%%%%s.%%%%f') AS event_time,
    `content_id`  AS content_id,
    `lesson_id`   AS lesson_id
FROM events_related_question_click
WHERE user_id IS NOT NULL
  AND user_id <> ''
  AND STR_TO_DATE(event_ts, '%%%%Y-%%%%m-%%%%d %%%%H:%%%%i:%%%%s.%%%%f') IS NOT NULL;
"""

execute_many(create_views_sql)

### VIEW 생성 여부 검증

In [ ]:
run_query("""
SELECT 'v_events_signup'        AS view_name, COUNT(*) AS row_cnt FROM v_events_signup
UNION ALL SELECT 'v_events_content_start',           COUNT(*) FROM v_events_content_start
UNION ALL SELECT 'v_events_lesson_view',       COUNT(*) FROM v_events_lesson_view
UNION ALL SELECT 'v_events_lesson_complete',         COUNT(*) FROM v_events_lesson_complete
UNION ALL SELECT 'v_events_content_end',             COUNT(*) FROM v_events_content_end
UNION ALL SELECT 'v_events_related_question_click',  COUNT(*) FROM v_events_related_question_click;
""", "view_check")

### 결측값 확인

In [ ]:
DATE_FMT = '%%%%Y-%%%%m-%%%%d %%%%H:%%%%i:%%%%s.%%%%f'

null_check_df = run_query(f"""
SELECT 'events_signup' AS table_name,
    COUNT(*) AS total,
    SUM(CASE WHEN NULLIF(TRIM(user_id), '') IS NULL THEN 1 ELSE 0 END) AS user_id_null,
    SUM(CASE WHEN STR_TO_DATE(event_ts, '{DATE_FMT}') IS NULL THEN 1 ELSE 0 END) AS event_time_null
FROM events_signup
UNION ALL
SELECT 'events_content_start', COUNT(*),
    SUM(CASE WHEN NULLIF(TRIM(user_id), '') IS NULL THEN 1 ELSE 0 END),
    SUM(CASE WHEN STR_TO_DATE(event_ts, '{DATE_FMT}') IS NULL THEN 1 ELSE 0 END)
FROM events_content_start
UNION ALL
SELECT 'events_lesson_view', COUNT(*),
    SUM(CASE WHEN NULLIF(TRIM(user_id), '') IS NULL THEN 1 ELSE 0 END),
    SUM(CASE WHEN STR_TO_DATE(event_ts, '{DATE_FMT}') IS NULL THEN 1 ELSE 0 END)
FROM events_lesson_view
UNION ALL
SELECT 'events_lesson_complete', COUNT(*),
    SUM(CASE WHEN NULLIF(TRIM(user_id), '') IS NULL THEN 1 ELSE 0 END),
    SUM(CASE WHEN STR_TO_DATE(event_ts, '{DATE_FMT}') IS NULL THEN 1 ELSE 0 END)
FROM events_lesson_complete
UNION ALL
SELECT 'events_content_end', COUNT(*),
    SUM(CASE WHEN NULLIF(TRIM(user_id), '') IS NULL THEN 1 ELSE 0 END),
    SUM(CASE WHEN STR_TO_DATE(event_ts, '{DATE_FMT}') IS NULL THEN 1 ELSE 0 END)
FROM events_content_end
UNION ALL
SELECT 'events_related_question_click', COUNT(*),
    SUM(CASE WHEN NULLIF(TRIM(user_id), '') IS NULL THEN 1 ELSE 0 END),
    SUM(CASE WHEN STR_TO_DATE(event_ts, '{DATE_FMT}') IS NULL THEN 1 ELSE 0 END)
FROM events_related_question_click;
""", "null_check")

null_check_df

### 중복값 확인

In [ ]:
duplicate_check_df = run_query("""
SELECT 'events_signup' AS table_name,
    COUNT(*) AS total,
    COUNT(DISTINCT user_id, event_time) AS unique_cnt,
    COUNT(*) - COUNT(DISTINCT user_id, event_time) AS duplicated_cnt
FROM v_events_signup
UNION ALL
SELECT 'events_content_start', COUNT(*), COUNT(DISTINCT user_id, event_time),
    COUNT(*) - COUNT(DISTINCT user_id, event_time)
FROM v_events_content_start
UNION ALL
SELECT 'events_lesson_view', COUNT(*), COUNT(DISTINCT user_id, event_time),
    COUNT(*) - COUNT(DISTINCT user_id, event_time)
FROM v_events_lesson_view
UNION ALL
SELECT 'events_lesson_complete', COUNT(*), COUNT(DISTINCT user_id, event_time),
    COUNT(*) - COUNT(DISTINCT user_id, event_time)
FROM v_events_lesson_complete
UNION ALL
SELECT 'events_content_end', COUNT(*), COUNT(DISTINCT user_id, event_time),
    COUNT(*) - COUNT(DISTINCT user_id, event_time)
FROM v_events_content_end
UNION ALL
SELECT 'events_related_question_click', COUNT(*),
    COUNT(DISTINCT user_id, event_time),
    COUNT(*) - COUNT(DISTINCT user_id, event_time)
FROM v_events_related_question_click;
""", "duplicate_check")

duplicate_check_df

### 이상치 1 : 시간 범위

In [ ]:
time_range_df = run_query("""
SELECT 'v_events_signup' AS view_name,
    MIN(event_time) AS min_t, MAX(event_time) AS max_t,
    SUM(CASE WHEN event_time > NOW() THEN 1 ELSE 0 END) AS future_cnt
FROM v_events_signup
UNION ALL
SELECT 'v_events_content_start', MIN(event_time), MAX(event_time),
    SUM(CASE WHEN event_time > NOW() THEN 1 ELSE 0 END)
FROM v_events_content_start
UNION ALL
SELECT 'v_events_lesson_view', MIN(event_time), MAX(event_time),
    SUM(CASE WHEN event_time > NOW() THEN 1 ELSE 0 END)
FROM v_events_lesson_view
UNION ALL
SELECT 'v_events_lesson_complete', MIN(event_time), MAX(event_time),
    SUM(CASE WHEN event_time > NOW() THEN 1 ELSE 0 END)
FROM v_events_lesson_complete
UNION ALL
SELECT 'v_events_content_end', MIN(event_time), MAX(event_time),
    SUM(CASE WHEN event_time > NOW() THEN 1 ELSE 0 END)
FROM v_events_content_end
UNION ALL
SELECT 'v_events_related_question_click', MIN(event_time), MAX(event_time),
    SUM(CASE WHEN event_time > NOW() THEN 1 ELSE 0 END)
FROM v_events_related_question_click;
""", "time_range")

time_range_df

### 이상치 2 : 가입 전 활동 (정합성)

In [ ]:
before_signup_df = run_query("""
WITH signup AS (
    SELECT user_id, MIN(event_time) AS signup_time
    FROM v_events_signup GROUP BY user_id
)
SELECT 'v_events_content_start' AS view_name,
    COUNT(*) AS before_signup_rows
FROM v_events_content_start sc
JOIN signup s ON sc.user_id = s.user_id
WHERE sc.event_time < s.signup_time
UNION ALL
SELECT 'v_events_lesson_view', COUNT(*)
FROM v_events_lesson_view l
JOIN signup s ON l.user_id = s.user_id
WHERE l.event_time < s.signup_time
UNION ALL
SELECT 'v_events_lesson_complete', COUNT(*)
FROM v_events_lesson_complete cl
JOIN signup s ON cl.user_id = s.user_id
WHERE cl.event_time < s.signup_time
UNION ALL
SELECT 'v_events_content_end', COUNT(*)
FROM v_events_content_end ec
JOIN signup s ON ec.user_id = s.user_id
WHERE ec.event_time < s.signup_time
UNION ALL
SELECT 'v_events_related_question_click', COUNT(*)
FROM v_events_related_question_click cq
JOIN signup s ON cq.user_id = s.user_id
WHERE cq.event_time < s.signup_time;
""", "before_signup")

before_signup_df

- 가입 전 콘텐츠 시작 : 11,920행
- 추측 원인 : 회원 가입 전 미리보기 / 데이터 소스 시계 동기화 차이 등

### 이상치 3 : 봇 의심 (한 유저가 너무 많은 이벤트)

- 유저별 일평균 활동량 분포 살펴보기

In [ ]:
user_stats_query = """
    SELECT
        user_id,
        COUNT(*) AS event_cnt,
        TIMESTAMPDIFF(DAY, MIN(event_time), MAX(event_time)) AS active_days,
        COUNT(*) / GREATEST(TIMESTAMPDIFF(DAY, MIN(event_time), MAX(event_time)), 1) AS daily_avg
    FROM v_events_lesson_view
    GROUP BY user_id
"""
df_user_stats = run_query(user_stats_query)

In [ ]:
print("전체 유저 수:", len(df_user_stats))
print("\n=== daily_avg 분포 ===")
print(df_user_stats['daily_avg'].describe(percentiles=[0.5, 0.9, 0.95, 0.99, 0.999]))

print("\n=== 임계값별 봇으로 분류되는 유저 수 ===")
for threshold in [20, 30, 50, 100, 200, 500, 974]:
    bot_count = (df_user_stats['daily_avg'] >= threshold).sum()
    pct = bot_count / len(df_user_stats) * 100
    print(f"  ≥ {threshold:>4}/일 : {bot_count:>6}명 ({pct:.3f}%)")

- 전체 11.4만 명

- 974/일 이상: 8명
- 500/일 이상: 93명
- 200/일 이상: 441명

In [ ]:
top_suspects = df_user_stats[df_user_stats['daily_avg'] >= 500].sort_values('daily_avg', ascending=False)
print(f"500/일 이상 유저: {len(top_suspects)}명")
print("\n=== 상위 20명 ===")
print(top_suspects[['user_id', 'event_cnt', 'active_days', 'daily_avg']].head(20))

print("\n=== 일평균 분포 (500+) ===")
print(top_suspects['daily_avg'].describe())

In [ ]:
sample_user = 'SAMPLE_USER_ID'

run_query(f"""
SELECT
    user_id,
    event_time,
    lesson_id,
    COUNT(*) AS dup_cnt
FROM v_events_lesson_view
WHERE user_id = '{sample_user}'
GROUP BY user_id, event_time, lesson_id
ORDER BY dup_cnt DESC
LIMIT 5;
""", "active0_check")

In [ ]:
THRESHOLD = 200
bot_candidates = df_user_stats[df_user_stats['daily_avg'] >= THRESHOLD].copy()
print(f"봇 후보: {len(bot_candidates)}명")

execute_many("""
DROP TABLE IF EXISTS bot_users;
CREATE TABLE bot_users (
    user_id     VARCHAR(64) NOT NULL PRIMARY KEY,
    event_cnt   INT,
    active_days INT,
    daily_avg   DECIMAL(10,2),
    detected_at DATETIME DEFAULT CURRENT_TIMESTAMP
) ENGINE=InnoDB;
""")

bot_candidates[['user_id', 'event_cnt', 'active_days', 'daily_avg']].to_sql(
    'bot_users', con=engine, if_exists='append', index=False
)

run_query("SELECT COUNT(*) AS cnt FROM bot_users", "bot_count")

# 연관 질문 클릭 구분

In [ ]:
create_events_content_end_view_sql = f"""
DROP VIEW IF EXISTS v_events_content_end;
CREATE VIEW v_events_content_end AS
SELECT
    ec.user_id,
    STR_TO_DATE(ec.event_ts, '{DATE_FMT}') AS event_time,
    ec.`content_id` AS content_id
FROM events_content_end ec
LEFT JOIN bot_users b ON ec.user_id = b.user_id
WHERE ec.user_id IS NOT NULL AND ec.user_id <> ''
  AND STR_TO_DATE(ec.event_ts, '{DATE_FMT}') IS NOT NULL
  AND b.user_id IS NULL;
"""

execute_many(create_events_content_end_view_sql)

In [ ]:
query = """
WITH
    signup AS (
        SELECT
            s.user_id,
            MIN(s.event_time) AS signup_time
        FROM v_events_signup s
        LEFT JOIN bot_users b ON s.user_id = b.user_id
        WHERE b.user_id IS NULL
        GROUP BY s.user_id
    ),

    first_content AS (
        SELECT
            sc.user_id,
            MIN(sc.event_time) AS first_content_time
        FROM v_events_content_start sc
        JOIN signup s ON sc.user_id = s.user_id AND sc.event_time >= s.signup_time
        GROUP BY sc.user_id
    ),

    first_lesson AS (
        SELECT
            el.user_id,
            MIN(el.event_time) AS first_lesson_time
        FROM v_events_lesson_view el
        JOIN first_content fc ON el.user_id = fc.user_id AND el.event_time >= fc.first_content_time
        GROUP BY el.user_id
    ),

    activation_users AS (
        SELECT
            cl.user_id,
            MIN(cl.event_time) AS activation_time
        FROM v_events_lesson_complete cl
        JOIN first_lesson fl ON cl.user_id = fl.user_id AND cl.event_time >= fl.first_lesson_time
        GROUP BY cl.user_id
    ),

    retention_users AS (
        SELECT
            au.user_id,
            au.activation_time
        FROM activation_users au
        JOIN v_events_lesson_view el ON au.user_id = el.user_id
           AND el.event_time >= DATE_ADD(au.activation_time, INTERVAL 24 HOUR)
           AND el.event_time < DATE_ADD(au.activation_time, INTERVAL 8 DAY)
        GROUP BY au.user_id, au.activation_time
    ),

    question_users AS (
        SELECT
            ru.user_id,
            MAX(CASE WHEN q.event_time IS NOT NULL THEN 1 ELSE 0 END) AS has_clicked_question
        FROM retention_users ru
        LEFT JOIN v_events_related_question_click q
            ON ru.user_id = q.user_id
           AND q.event_time >= ru.activation_time
           AND q.event_time < DATE_ADD(ru.activation_time, INTERVAL 8 DAY)
        GROUP BY ru.user_id
    )

SELECT
    CASE WHEN has_clicked_question = 1 THEN '질문 클릭' ELSE '미클릭' END AS question_status,
    COUNT(DISTINCT user_id) AS user_count,
    ROUND(COUNT(DISTINCT user_id) / SUM(COUNT(DISTINCT user_id)) OVER () * 100, 1) AS pct
FROM question_users
GROUP BY 1
ORDER BY user_count DESC;
"""

question_df = pd.read_sql(query, engine)
question_df

# 평균 레슨 완료 수

In [ ]:
query = """
WITH
    signup AS (
        SELECT
            s.user_id,
            MIN(s.event_time) AS signup_time
        FROM v_events_signup s
        LEFT JOIN bot_users b ON s.user_id = b.user_id
        WHERE b.user_id IS NULL
        GROUP BY s.user_id
    ),

    first_content AS (
        SELECT
            sc.user_id,
            MIN(sc.event_time) AS first_content_time
        FROM v_events_content_start sc
        JOIN signup s
            ON sc.user_id = s.user_id
           AND sc.event_time >= s.signup_time
        GROUP BY sc.user_id
    ),

    first_lesson AS (
        SELECT
            el.user_id,
            MIN(el.event_time) AS first_lesson_time
        FROM v_events_lesson_view el
        JOIN first_content fc
            ON el.user_id = fc.user_id
           AND el.event_time >= fc.first_content_time
        GROUP BY el.user_id
    ),

    activation_users AS (
        SELECT
            cl.user_id,
            MIN(cl.event_time) AS activation_time
        FROM v_events_lesson_complete cl
        JOIN first_lesson fl
            ON cl.user_id = fl.user_id
           AND cl.event_time >= fl.first_lesson_time
        GROUP BY cl.user_id
    ),

    retention_users AS (
        SELECT
            au.user_id,
            au.activation_time
        FROM activation_users au
        JOIN v_events_lesson_view el
            ON au.user_id = el.user_id
           AND el.event_time >= DATE_ADD(au.activation_time, INTERVAL 24 HOUR)
           AND el.event_time < DATE_ADD(au.activation_time, INTERVAL 8 DAY)
        GROUP BY au.user_id, au.activation_time
    ),

    question_users AS (
        SELECT
            ru.user_id,
            ru.activation_time,
            MAX(CASE WHEN q.event_time IS NOT NULL THEN 1 ELSE 0 END) AS has_clicked_question
        FROM retention_users ru
        LEFT JOIN v_events_related_question_click q
            ON ru.user_id = q.user_id
           AND q.event_time >= ru.activation_time
           AND q.event_time < DATE_ADD(ru.activation_time, INTERVAL 8 DAY)
        GROUP BY ru.user_id, ru.activation_time
    ),

    lesson_count AS (
        SELECT
            qu.user_id,
            qu.has_clicked_question,
            COUNT(cl.event_time) AS lessons_completed
        FROM question_users qu
        LEFT JOIN v_events_lesson_complete cl
            ON qu.user_id = cl.user_id
           AND cl.event_time >= qu.activation_time
           AND cl.event_time < DATE_ADD(qu.activation_time, INTERVAL 8 DAY)
        GROUP BY qu.user_id, qu.has_clicked_question
    )

SELECT
    CASE
        WHEN has_clicked_question = 1 THEN '질문 클릭'
        ELSE '미클릭'
    END AS question_status,
    COUNT(DISTINCT user_id) AS user_count,
    AVG(lessons_completed) AS avg_lessons_completed,
    MIN(lessons_completed) AS min_lessons_completed,
    MAX(lessons_completed) AS max_lessons_completed,
    SUM(lessons_completed) AS total_lessons_completed
FROM lesson_count
GROUP BY question_status
ORDER BY question_status;
"""

lesson_by_question_original_df = pd.read_sql(query, engine)
lesson_by_question_original_df

In [ ]:
query = """
WITH
    signup AS (
        SELECT
            s.user_id,
            MIN(s.event_time) AS signup_time
        FROM v_events_signup s
        LEFT JOIN bot_users b ON s.user_id = b.user_id
        WHERE b.user_id IS NULL
        GROUP BY s.user_id
    ),

    first_content AS (
        SELECT
            sc.user_id,
            MIN(sc.event_time) AS first_content_time
        FROM v_events_content_start sc
        JOIN signup s
            ON sc.user_id = s.user_id
           AND sc.event_time >= s.signup_time
        GROUP BY sc.user_id
    ),

    first_lesson AS (
        SELECT
            el.user_id,
            MIN(el.event_time) AS first_lesson_time
        FROM v_events_lesson_view el
        JOIN first_content fc
            ON el.user_id = fc.user_id
           AND el.event_time >= fc.first_content_time
        GROUP BY el.user_id
    ),

    activation_users AS (
        SELECT
            cl.user_id,
            MIN(cl.event_time) AS activation_time
        FROM v_events_lesson_complete cl
        JOIN first_lesson fl
            ON cl.user_id = fl.user_id
           AND cl.event_time >= fl.first_lesson_time
        GROUP BY cl.user_id
    ),

    retention_users AS (
        SELECT
            au.user_id,
            au.activation_time
        FROM activation_users au
        JOIN v_events_lesson_view el
            ON au.user_id = el.user_id
           AND el.event_time >= DATE_ADD(au.activation_time, INTERVAL 24 HOUR)
           AND el.event_time < DATE_ADD(au.activation_time, INTERVAL 8 DAY)
        GROUP BY au.user_id, au.activation_time
    ),

    question_users AS (
        SELECT
            ru.user_id,
            ru.activation_time,
            MAX(CASE WHEN q.event_time IS NOT NULL THEN 1 ELSE 0 END) AS has_clicked_question
        FROM retention_users ru
        LEFT JOIN v_events_related_question_click q
            ON ru.user_id = q.user_id
           AND q.event_time >= ru.activation_time
           AND q.event_time < DATE_ADD(ru.activation_time, INTERVAL 8 DAY)
        GROUP BY ru.user_id, ru.activation_time
    ),

    lesson_count AS (
        SELECT
            qu.user_id,
            qu.has_clicked_question,
            COUNT(cl.event_time) AS lessons_completed
        FROM question_users qu
        LEFT JOIN v_events_lesson_complete cl
            ON qu.user_id = cl.user_id
           AND cl.event_time >= qu.activation_time
           AND cl.event_time < DATE_ADD(qu.activation_time, INTERVAL 8 DAY)
        GROUP BY qu.user_id, qu.has_clicked_question
    ),

    ranked AS (
        SELECT
            *,
            ROW_NUMBER() OVER (
                PARTITION BY has_clicked_question
                ORDER BY lessons_completed
            ) AS rn,

            COUNT(*) OVER (
                PARTITION BY has_clicked_question
            ) AS cnt
        FROM lesson_count
    )

SELECT
    CASE
        WHEN has_clicked_question = 1 THEN '질문 클릭'
        ELSE '미클릭'
    END AS question_status,

    ROUND(AVG(lessons_completed), 2) AS median_lessons_completed

FROM ranked
WHERE rn IN (
    FLOOR((cnt + 1) / 2),
    FLOOR((cnt + 2) / 2)
)
GROUP BY has_clicked_question
ORDER BY has_clicked_question;
"""

median_df = pd.read_sql(query, engine)
median_df

# 콘텐츠 완강률

In [ ]:
query = """
WITH
    signup AS (
        SELECT
            s.user_id,
            MIN(s.event_time) AS signup_time
        FROM v_events_signup s
        LEFT JOIN bot_users b ON s.user_id = b.user_id
        WHERE b.user_id IS NULL
        GROUP BY s.user_id
    ),

    first_content AS (
        SELECT
            sc.user_id,
            MIN(sc.event_time) AS first_content_time
        FROM v_events_content_start sc
        JOIN signup s ON sc.user_id = s.user_id AND sc.event_time >= s.signup_time
        GROUP BY sc.user_id
    ),

    first_lesson AS (
        SELECT
            el.user_id,
            MIN(el.event_time) AS first_lesson_time
        FROM v_events_lesson_view el
        JOIN first_content fc ON el.user_id = fc.user_id AND el.event_time >= fc.first_content_time
        GROUP BY el.user_id
    ),

    activation_users AS (
        SELECT
            cl.user_id,
            MIN(cl.event_time) AS activation_time
        FROM v_events_lesson_complete cl
        JOIN first_lesson fl ON cl.user_id = fl.user_id AND cl.event_time >= fl.first_lesson_time
        GROUP BY cl.user_id
    ),

    retention_users AS (
        SELECT
            au.user_id,
            au.activation_time
        FROM activation_users au
        JOIN v_events_lesson_view el ON au.user_id = el.user_id
           AND el.event_time >= DATE_ADD(au.activation_time, INTERVAL 24 HOUR)
           AND el.event_time < DATE_ADD(au.activation_time, INTERVAL 8 DAY)
        GROUP BY au.user_id, au.activation_time
    ),

    question_users AS (
        SELECT
            ru.user_id,
            MAX(CASE WHEN q.event_time IS NOT NULL THEN 1 ELSE 0 END) AS has_clicked_question
        FROM retention_users ru
        LEFT JOIN v_events_related_question_click q
            ON ru.user_id = q.user_id
           AND q.event_time >= ru.activation_time
           AND q.event_time < DATE_ADD(ru.activation_time, INTERVAL 8 DAY)
        GROUP BY ru.user_id
    ),

    additional_completed_users AS (
        SELECT
            r.user_id,
            r.activation_time,
            MIN(cl.event_time) AS additional_complete_time
        FROM retention_users r
        JOIN v_events_lesson_complete cl
            ON r.user_id = cl.user_id
           -- retention_time이 따로 없었으므로, 리텐션 기준과 동일하게 활성화 후 24시간 이후로 잡았어
           AND cl.event_time >= DATE_ADD(r.activation_time, INTERVAL 24 HOUR)
           AND cl.event_time < DATE_ADD(r.activation_time, INTERVAL 8 DAY)
        GROUP BY r.user_id, r.activation_time
    ),

    content_completed_users AS (
        SELECT DISTINCT
            ac.user_id
        FROM additional_completed_users ac
        JOIN v_events_content_end ec
            ON ac.user_id = ec.user_id
           AND ec.event_time >= ac.additional_complete_time
           AND ec.event_time < DATE_ADD(ac.activation_time, INTERVAL 8 DAY)
    )

SELECT
    CASE WHEN q.has_clicked_question = 1 THEN '질문 클릭' ELSE '미클릭' END AS question_status,
    COUNT(DISTINCT q.user_id) AS total_users,
    COUNT(DISTINCT c.user_id) AS completed_users,
    ROUND(COUNT(DISTINCT c.user_id) / COUNT(DISTINCT q.user_id) * 100, 1) AS completion_rate_pct
FROM question_users q
LEFT JOIN content_completed_users c
    ON q.user_id = c.user_id
GROUP BY 1
ORDER BY total_users DESC;
"""

content_question_df = pd.read_sql(query, engine)
content_question_df

# Revenue 전환율

In [ ]:
DATE_FMT = '%%%%Y-%%%%m-%%%%d %%%%H:%%%%i:%%%%s.%%%%f'

create_revenue_views_sql = f"""
DROP VIEW IF EXISTS v_events_payment_page_view;
CREATE VIEW v_events_payment_page_view AS
SELECT
    epp.user_id,
    STR_TO_DATE(epp.event_ts, '{DATE_FMT}') AS event_time
FROM events_payment_page_view epp
LEFT JOIN bot_users b ON epp.user_id = b.user_id
WHERE epp.user_id IS NOT NULL AND epp.user_id <> ''
  AND STR_TO_DATE(epp.event_ts, '{DATE_FMT}') IS NOT NULL
  AND b.user_id IS NULL;


DROP VIEW IF EXISTS v_events_subscription_complete;
CREATE VIEW v_events_subscription_complete AS
SELECT
    cs.user_id,
    STR_TO_DATE(cs.event_ts, '{DATE_FMT}') AS event_time,
    cs.paid_amount,
    cs.`plan_price`              AS plan_price,
    cs.`discount_amount`  AS coupon_discount_amount,
    cs.`payment_method`                 AS pg_type
FROM events_subscription_complete cs
LEFT JOIN bot_users b ON cs.user_id = b.user_id
WHERE cs.user_id IS NOT NULL AND cs.user_id <> ''
  AND STR_TO_DATE(cs.event_ts, '{DATE_FMT}') IS NOT NULL
  AND b.user_id IS NULL;


DROP VIEW IF EXISTS v_events_subscription_renew;
CREATE VIEW v_events_subscription_renew AS
SELECT
    rs.user_id,
    STR_TO_DATE(rs.event_ts, '{DATE_FMT}') AS event_time,
    rs.paid_amount,
    rs.`plan_price`              AS plan_price,
    rs.`discount_amount`  AS coupon_discount_amount,
    rs.`payment_method`                 AS pg_type
FROM events_subscription_renew rs
LEFT JOIN bot_users b ON rs.user_id = b.user_id
WHERE rs.user_id IS NOT NULL AND rs.user_id <> ''
  AND STR_TO_DATE(rs.event_ts, '{DATE_FMT}') IS NOT NULL
  AND b.user_id IS NULL;


DROP VIEW IF EXISTS v_events_subscription_resubscribe;
CREATE VIEW v_events_subscription_resubscribe AS
SELECT
    rss.user_id,
    STR_TO_DATE(rss.event_ts, '{DATE_FMT}') AS event_time,
    rss.paid_amount,
    rss.`plan_price`              AS plan_price,
    rss.`discount_amount`  AS coupon_discount_amount,
    rss.`payment_method`                 AS pg_type
FROM events_subscription_resubscribe rss
LEFT JOIN bot_users b ON rss.user_id = b.user_id
WHERE rss.user_id IS NOT NULL AND rss.user_id <> ''
  AND STR_TO_DATE(rss.event_ts, '{DATE_FMT}') IS NOT NULL
  AND b.user_id IS NULL;


DROP VIEW IF EXISTS v_events_trial_start;
CREATE VIEW v_events_trial_start AS
SELECT
    ft.user_id,
    STR_TO_DATE(ft.event_ts, '{DATE_FMT}') AS event_time,
    ft.`plan_price` AS plan_price,
    ft.`plan_type`  AS plan_type
FROM events_trial_start ft
LEFT JOIN bot_users b ON ft.user_id = b.user_id
WHERE ft.user_id IS NOT NULL AND ft.user_id <> ''
  AND STR_TO_DATE(ft.event_ts, '{DATE_FMT}') IS NOT NULL
  AND b.user_id IS NULL;
"""

execute_many(create_revenue_views_sql)

# 결제 퍼널

In [ ]:
query = '''
WITH
    signup AS (
        SELECT
            s.user_id,
            MIN(s.event_time) AS signup_time
        FROM v_events_signup s
        LEFT JOIN bot_users b
            ON s.user_id = b.user_id
        WHERE b.user_id IS NULL
        GROUP BY s.user_id
    ),

    first_content AS (
        SELECT
            sc.user_id,
            MIN(sc.event_time) AS first_content_time
        FROM v_events_content_start sc
        JOIN signup s
            ON sc.user_id = s.user_id
           AND sc.event_time >= s.signup_time
        GROUP BY sc.user_id
    ),

    first_lesson AS (
        SELECT
            el.user_id,
            MIN(el.event_time) AS first_lesson_time
        FROM v_events_lesson_view el
        JOIN first_content fc
            ON el.user_id = fc.user_id
           AND el.event_time >= fc.first_content_time
        GROUP BY el.user_id
    ),

    activation_users AS (
        SELECT
            cl.user_id,
            MIN(cl.event_time) AS activation_time
        FROM v_events_lesson_complete cl
        JOIN first_lesson fl
            ON cl.user_id = fl.user_id
           AND cl.event_time >= fl.first_lesson_time
        GROUP BY cl.user_id
    ),

    retained_users AS (
        SELECT
            a.user_id,
            MIN(el.event_time) AS retention_time
        FROM activation_users a
        JOIN v_events_lesson_view el
            ON a.user_id = el.user_id
           AND el.event_time >= DATE_ADD(a.activation_time, INTERVAL 24 HOUR)
           AND el.event_time <  DATE_ADD(a.activation_time, INTERVAL 8 DAY)
        GROUP BY a.user_id
    ),

    payment_page_users AS (
        SELECT
            r.user_id,
            MIN(pp.event_time) AS payment_page_time
        FROM retained_users r
        JOIN v_events_payment_page_view pp
            ON r.user_id = pp.user_id
           AND pp.event_time >= r.retention_time -- 리텐션 발생 이후 결제창 진입만
        GROUP BY r.user_id
    ),

    subscribed_users AS (
        SELECT DISTINCT
            pp.user_id
        FROM payment_page_users pp
        JOIN v_events_subscription_complete cs
            ON pp.user_id = cs.user_id
           AND cs.event_time >= pp.payment_page_time -- 결제창 진입 이후 구독만
    )

SELECT
    COUNT(DISTINCT r.user_id) AS retention_users,
    COUNT(DISTINCT pp.user_id) AS payment_page_users,
    COUNT(DISTINCT s.user_id) AS subscribed_users,

    COUNT(DISTINCT r.user_id) - COUNT(DISTINCT pp.user_id) AS retention_to_payment_dropout_users,
    COUNT(DISTINCT pp.user_id) - COUNT(DISTINCT s.user_id) AS payment_to_subscription_dropout_users,

    ROUND(
        COUNT(DISTINCT pp.user_id) * 100.0 / NULLIF(COUNT(DISTINCT r.user_id), 0),
        2
    ) AS retention_to_payment_pct,

    ROUND(
        COUNT(DISTINCT s.user_id) * 100.0 / NULLIF(COUNT(DISTINCT pp.user_id), 0),
        2
    ) AS payment_to_subscription_pct,

    ROUND(
        COUNT(DISTINCT s.user_id) * 100.0 / NULLIF(COUNT(DISTINCT r.user_id), 0),
        2
    ) AS retention_to_subscription_pct

FROM retained_users r
LEFT JOIN payment_page_users pp
    ON r.user_id = pp.user_id
LEFT JOIN subscribed_users s
    ON pp.user_id = s.user_id;
'''

revenue_funnel_df = pd.read_sql(query, engine)
revenue_funnel_df

# 결제 페이지 진입율

In [ ]:
query = """
WITH
    signup AS (
        SELECT
            s.user_id,
            MIN(s.event_time) AS signup_time
        FROM v_events_signup s
        LEFT JOIN bot_users b
            ON s.user_id = b.user_id
        WHERE b.user_id IS NULL
        GROUP BY s.user_id
    ),

    first_content AS (
        SELECT
            sc.user_id,
            MIN(sc.event_time) AS first_content_time
        FROM v_events_content_start sc
        JOIN signup s
            ON sc.user_id = s.user_id
           AND sc.event_time >= s.signup_time
        GROUP BY sc.user_id
    ),

    first_lesson AS (
        SELECT
            el.user_id,
            MIN(el.event_time) AS first_lesson_time
        FROM v_events_lesson_view el
        JOIN first_content fc
            ON el.user_id = fc.user_id
           AND el.event_time >= fc.first_content_time
        GROUP BY el.user_id
    ),

    activation_users AS (
        SELECT
            cl.user_id,
            MIN(cl.event_time) AS activation_time
        FROM v_events_lesson_complete cl
        JOIN first_lesson fl
            ON cl.user_id = fl.user_id
           AND cl.event_time >= fl.first_lesson_time
        GROUP BY cl.user_id
    ),

    retained_users AS (
        SELECT
            au.user_id,
            au.activation_time,
            MIN(el.event_time) AS retention_time
        FROM activation_users au
        JOIN v_events_lesson_view el
            ON au.user_id = el.user_id
           AND el.event_time >= DATE_ADD(au.activation_time, INTERVAL 24 HOUR)
           AND el.event_time <  DATE_ADD(au.activation_time, INTERVAL 8 DAY)
        GROUP BY
            au.user_id,
            au.activation_time
    ),

    question_groups AS (
        SELECT
            ru.user_id,
            ru.activation_time,
            ru.retention_time,
            CASE
                WHEN COUNT(q.event_time) > 0 THEN '질문 클릭'
                ELSE '미클릭'
            END AS question_status
        FROM retained_users ru
        LEFT JOIN v_events_related_question_click q
            ON ru.user_id = q.user_id
           AND q.event_time >= ru.activation_time
           AND q.event_time <  DATE_ADD(ru.activation_time, INTERVAL 8 DAY)
        GROUP BY
            ru.user_id,
            ru.activation_time,
            ru.retention_time
    ),

    payment_page_users AS (
        SELECT
            qg.user_id,
            MIN(pp.event_time) AS payment_page_time
        FROM question_groups qg
        JOIN v_events_payment_page_view pp
            ON qg.user_id = pp.user_id
           AND pp.event_time >= qg.retention_time
        GROUP BY qg.user_id
    )

SELECT
    qg.question_status AS 구분,
    COUNT(DISTINCT qg.user_id) AS `잔존 유저 수`,
    COUNT(DISTINCT ppu.user_id) AS `결제 페이지 진입자`,
    ROUND(
        COUNT(DISTINCT ppu.user_id) * 100.0 / COUNT(DISTINCT qg.user_id),
        2
    ) AS `결제 페이지 진입률`
FROM question_groups qg
LEFT JOIN payment_page_users ppu
    ON qg.user_id = ppu.user_id
GROUP BY qg.question_status
ORDER BY `결제 페이지 진입률` DESC;
"""

question_payment_df = pd.read_sql(query, engine)
question_payment_df

# 페이지 진입 및 구독 전환율

In [ ]:
query = '''
WITH
    signup AS (
        SELECT s.user_id, MIN(s.event_time) AS signup_time
        FROM v_events_signup s
        LEFT JOIN bot_users b ON s.user_id = b.user_id
        WHERE b.user_id IS NULL
        GROUP BY s.user_id
    ),
    first_content AS (
        SELECT sc.user_id, MIN(sc.event_time) AS first_content_time
        FROM v_events_content_start sc
        JOIN signup s ON sc.user_id = s.user_id AND sc.event_time >= s.signup_time
        GROUP BY sc.user_id
    ),
    first_lesson AS (
        SELECT el.user_id, MIN(el.event_time) AS first_lesson_time
        FROM v_events_lesson_view el
        JOIN first_content fc ON el.user_id = fc.user_id AND el.event_time >= fc.first_content_time
        GROUP BY el.user_id
    ),
    activation_users AS (
        SELECT cl.user_id, MIN(cl.event_time) AS activation_time
        FROM v_events_lesson_complete cl
        JOIN first_lesson fl ON cl.user_id = fl.user_id AND cl.event_time >= fl.first_lesson_time
        GROUP BY cl.user_id
    ),

    retention_users AS (
        SELECT
            au.user_id,
            au.activation_time,
            MIN(el.event_time) AS retention_time
        FROM activation_users au
        JOIN v_events_lesson_view el ON au.user_id = el.user_id
           AND el.event_time >= DATE_ADD(au.activation_time, INTERVAL 24 HOUR)
           AND el.event_time < DATE_ADD(au.activation_time, INTERVAL 8 DAY)
        GROUP BY au.user_id, au.activation_time
    ),

    question_clickers AS (
        SELECT DISTINCT ru.user_id, ru.retention_time
        FROM retention_users ru
        JOIN v_events_related_question_click q
            ON ru.user_id = q.user_id
           AND q.event_time >= ru.activation_time
           AND q.event_time < DATE_ADD(ru.activation_time, INTERVAL 8 DAY)
    ),

    payment_page_users AS (
        SELECT DISTINCT ru.user_id
        FROM retention_users ru
        JOIN v_events_payment_page_view pp
            ON ru.user_id = pp.user_id
           AND pp.event_time >= ru.retention_time
    ),

    subscribed_users AS (
        SELECT DISTINCT ru.user_id
        FROM retention_users ru
        JOIN v_events_subscription_complete cs
            ON ru.user_id = cs.user_id
           AND cs.event_time >= ru.retention_time
    )

SELECT
    CASE WHEN qc.user_id IS NOT NULL THEN '질문 클릭' ELSE '미클릭' END AS group_type,
    COUNT(DISTINCT ru.user_id) AS total_users,
    COUNT(DISTINCT pp.user_id) AS payment_page_users,
    COUNT(DISTINCT su.user_id) AS subscribed_users,

    ROUND(COUNT(DISTINCT pp.user_id) * 100.0 / COUNT(DISTINCT ru.user_id), 2)            AS payment_entry_pct,
    ROUND(COUNT(DISTINCT su.user_id) * 100.0 / NULLIF(COUNT(DISTINCT pp.user_id), 0), 2) AS payment_to_sub_pct,
    ROUND(COUNT(DISTINCT su.user_id) * 100.0 / COUNT(DISTINCT ru.user_id), 2)            AS total_conversion_pct

FROM retention_users ru
LEFT JOIN question_clickers   qc ON ru.user_id = qc.user_id
LEFT JOIN payment_page_users  pp ON ru.user_id = pp.user_id
LEFT JOIN subscribed_users    su ON ru.user_id = su.user_id
GROUP BY group_type
ORDER BY group_type DESC;
'''

clicker_vs_non_df = pd.read_sql(query, engine)
clicker_vs_non_df

In [ ]:
metrics = ['payment_entry_pct', 'payment_to_sub_pct', 'total_conversion_pct']
metric_labels = ['결제 페이지 진입률 (%)', '결제→구독 전환율 (%)', '최종 구독 전환율 (%)']

x = np.arange(len(metric_labels))
width = 0.35

clicker_data = clicker_vs_non_df[
    clicker_vs_non_df['group_type'] == '질문 클릭'
][metrics].values.flatten()

non_clicker_data = clicker_vs_non_df[
    clicker_vs_non_df['group_type'] == '미클릭'
][metrics].values.flatten()

fig, ax = plt.subplots(figsize=(11, 6))
rects1 = ax.bar(x - width/2, clicker_data, width,
                label='질문 클릭 (8,018명)', color='#2171b5')
rects2 = ax.bar(x + width/2, non_clicker_data, width,
                label='미클릭 (3,721명)', color='#bdd8f1')

def autolabel(rects):
    for rect in rects:
        height = rect.get_height()
        if pd.notnull(height):
            ax.annotate(f'{height:.1f}%',
                        xy=(rect.get_x() + rect.get_width() / 2, height),
                        xytext=(0, 5), textcoords='offset points',
                        ha='center', fontsize=11, fontweight='bold')

autolabel(rects1)
autolabel(rects2)

for i, (c, n) in enumerate(zip(clicker_data, non_clicker_data)):
    if pd.notnull(c) and pd.notnull(n):
        diff = c - n
        color = '#C44E52' if diff > 0 else '#2C7BB6'
        ax.annotate(f'+{diff:.1f}%p',
                    xy=(i, max(c, n) + 5),
                    ha='center', fontsize=10,
                    color=color, fontweight='bold')

ax.set_ylabel('전환율 (%)', fontsize=12)
ax.set_title('재진입 이후 질문 클릭 여부별 결제 전환율 비교\n(Retention 유저 11,739명 기준)',
             fontsize=14, fontweight='bold', pad=15)
ax.set_xticks(x)
ax.set_xticklabels(metric_labels, fontsize=11)
ax.set_ylim(0, max(clicker_data.max(), non_clicker_data.max()) * 1.25)
ax.legend(fontsize=11, loc='upper right')
ax.spines[['top', 'right']].set_visible(False)
ax.grid(axis='y', linestyle='--', alpha=0.3)

plt.tight_layout()
plt.show()

# ARPU / ARPPU

In [ ]:
query_question_user = """
WITH
    signup AS (
        SELECT
            s.user_id,
            MIN(s.event_time) AS signup_time
        FROM v_events_signup s
        LEFT JOIN bot_users b ON s.user_id = b.user_id
        WHERE b.user_id IS NULL
        GROUP BY s.user_id
    ),

    first_content AS (
        SELECT
            sc.user_id,
            MIN(sc.event_time) AS first_content_time
        FROM v_events_content_start sc
        JOIN signup s
            ON sc.user_id = s.user_id
           AND sc.event_time >= s.signup_time
        GROUP BY sc.user_id
    ),

    first_lesson AS (
        SELECT
            el.user_id,
            MIN(el.event_time) AS first_lesson_time
        FROM v_events_lesson_view el
        JOIN first_content fc
            ON el.user_id = fc.user_id
           AND el.event_time >= fc.first_content_time
        GROUP BY el.user_id
    ),

    activation_users AS (
        SELECT
            cl.user_id,
            MIN(cl.event_time) AS activation_time
        FROM v_events_lesson_complete cl
        JOIN first_lesson fl
            ON cl.user_id = fl.user_id
           AND cl.event_time >= fl.first_lesson_time
        GROUP BY cl.user_id
    ),

    retention_users AS (
        SELECT
            au.user_id,
            au.activation_time
        FROM activation_users au
        JOIN v_events_lesson_view el
            ON au.user_id = el.user_id
           AND el.event_time >= DATE_ADD(au.activation_time, INTERVAL 24 HOUR)
           AND el.event_time < DATE_ADD(au.activation_time, INTERVAL 8 DAY)
        GROUP BY au.user_id, au.activation_time
    ),

    question_users AS (
        SELECT
            ru.user_id,
            MAX(CASE WHEN q.event_time IS NOT NULL THEN 1 ELSE 0 END) AS has_clicked_question
        FROM retention_users ru
        LEFT JOIN v_events_related_question_click q
            ON ru.user_id = q.user_id
           AND q.event_time >= ru.activation_time
           AND q.event_time < DATE_ADD(ru.activation_time, INTERVAL 8 DAY)
        GROUP BY ru.user_id
    )

SELECT
    user_id,
    CASE
        WHEN has_clicked_question = 1 THEN '질문 클릭'
        ELSE '미클릭'
    END AS question_status
FROM question_users
"""

question_user_df = pd.read_sql(query_question_user, engine)
question_user_df

In [ ]:
query_payment = """
SELECT
    user_id,
    SUM(CAST(REPLACE(`plan_price`, ',', '') AS DECIMAL(18,2))) AS revenue
FROM events_subscription_complete
WHERE NULLIF(user_id, '') IS NOT NULL
  AND NULLIF(`plan_price`, '') IS NOT NULL
  AND CAST(REPLACE(`plan_price`, ',', '') AS DECIMAL(18,2)) IN (
      15920, 131600, 79200, 95520, 83200, 42960, 14328, 118440
  )
GROUP BY user_id
"""

payment_df = pd.read_sql(query_payment, engine)

revenue_base = question_user_df.merge(
    payment_df,
    on='user_id',
    how='left'
)

revenue_base['revenue'] = revenue_base['revenue'].fillna(0)

revenue_summary = (
    revenue_base
    .groupby('question_status')
    .agg(
        분석대상_유저수=('user_id', 'nunique'),
        구독_유저수=('revenue', lambda x: (x > 0).sum()),
        총수익=('revenue', 'sum')
    )
    .reset_index()
)

revenue_summary['ARPU'] = (
    revenue_summary['총수익'] / revenue_summary['분석대상_유저수']
).round(0)

revenue_summary['ARPPU'] = (
    revenue_summary['총수익'] / revenue_summary['구독_유저수']
).round(0)

revenue_summary['총수익_억_원'] = (
    revenue_summary['총수익'] / 100000000
).round(2)

revenue_summary = revenue_summary.rename(columns={
    'question_status': '구분',
    '분석대상_유저수': '분석 대상 유저 수 (명)',
    '구독_유저수': '구독 유저 수 (명)',
    '총수익_억_원': '총 수익 (억 원)',
    'ARPU': 'ARPU (원)',
    'ARPPU': 'ARPPU (원)'
})

revenue_summary = revenue_summary[
    [
        '구분',
        '분석 대상 유저 수 (명)',
        '구독 유저 수 (명)',
        '총 수익 (억 원)',
        'ARPU (원)',
        'ARPPU (원)'
    ]
]

revenue_summary

In [ ]:
query_payment = """
SELECT
    user_id,
    SUM(CAST(REPLACE(`plan_price`, ',', '') AS DECIMAL(18,2))) AS revenue
FROM events_subscription_complete
WHERE NULLIF(user_id, '') IS NOT NULL
  AND NULLIF(`plan_price`, '') IS NOT NULL
  AND CAST(REPLACE(`plan_price`, ',', '') AS DECIMAL(18,2)) IN (
      15920, 131600, 79200, 95520, 83200, 42960, 14328, 118440
  )
GROUP BY user_id
"""

payment_df = pd.read_sql(query_payment, engine)

arppu_base = question_user_df.merge(
    payment_df,
    on='user_id',
    how='left'
)

arppu_base['revenue'] = arppu_base['revenue'].fillna(0)

arppu_result = (
    arppu_base
    .groupby('question_status')
    .agg(
        total_users=('user_id', 'nunique'),
        paying_users=('revenue', lambda x: (x > 0).sum()),
        total_revenue=('revenue', 'sum')
    )
    .reset_index()
)

arppu_result['arppu'] = (
    arppu_result['total_revenue'] / arppu_result['paying_users']
).round(0)

arppu_result['paying_user_pct'] = (
    arppu_result['paying_users'] / arppu_result['total_users'] * 100
).round(1)

arppu_result['total_revenue'] = arppu_result['total_revenue'].round(0).astype(int)
arppu_result['arppu'] = arppu_result['arppu'].fillna(0).astype(int)

arppu_result

In [ ]:
query_payment = """
SELECT
    user_id,
    SUM(CAST(REPLACE(`plan_price`, ',', '') AS DECIMAL(18,2))) AS revenue
FROM events_subscription_complete
WHERE NULLIF(user_id, '') IS NOT NULL
  AND NULLIF(`plan_price`, '') IS NOT NULL
  AND CAST(REPLACE(`plan_price`, ',', '') AS DECIMAL(18,2)) IN (
      15920, 131600, 79200, 95520, 83200, 42960, 14328, 118440
  )
GROUP BY user_id
"""

payment_df = pd.read_sql(query_payment, engine)

revenue_base = question_user_df.merge(
    payment_df,
    on='user_id',
    how='left'
)

revenue_base['revenue'] = revenue_base['revenue'].fillna(0)

revenue_contribution_df = (
    revenue_base
    .groupby('question_status')
    .agg(
        total_users=('user_id', 'nunique'),
        paying_users=('revenue', lambda x: (x > 0).sum()),
        total_revenue=('revenue', 'sum')
    )
    .reset_index()
)

revenue_contribution_df['revenue_contribution_pct'] = (
    revenue_contribution_df['total_revenue']
    / revenue_contribution_df['total_revenue'].sum()
    * 100
).round(1)

revenue_contribution_df['total_revenue'] = (
    revenue_contribution_df['total_revenue']
    .round(0)
    .astype(int)
)

revenue_contribution_df = revenue_contribution_df.sort_values(
    'total_revenue',
    ascending=False
)

revenue_contribution_df

# 데이터 추출

In [ ]:
!pip install openpyxl

In [ ]:
file_path = 'tableau_dashboard_data.xlsx'

with pd.ExcelWriter(file_path, engine='openpyxl') as writer:
    question_df.to_excel(writer, sheet_name='question_data', index=False)
    lesson_by_question_original_df.to_excel(writer, sheet_name='lesson_by_q', index=False)
    median_df.to_excel(writer, sheet_name='median_data', index=False)
    content_question_df.to_excel(writer, sheet_name='content_q', index=False)
    revenue_funnel_df.to_excel(writer, sheet_name='revenue_funnel', index=False)
    question_payment_df.to_excel(writer, sheet_name='question_payment', index=False)
    clicker_vs_non_df.to_excel(writer, sheet_name='clicker_vs_non', index=False)
    question_user_df.to_excel(writer, sheet_name='question_user', index=False)
    revenue_summary.to_excel(writer, sheet_name='revenue_summary', index=False)
    arppu_result.to_excel(writer, sheet_name='arppu_result', index=False)
    revenue_contribution_df.to_excel(writer, sheet_name='rev_contribution', index=False)

print(f"✅ '{file_path}' 내보내기 완료!")

In [ ]:
import os

folder_name = 'tableau_csv_data'
os.makedirs(folder_name, exist_ok=True)

question_df.to_csv(f'{folder_name}/question_data.csv', index=False, encoding='utf-8-sig')
lesson_by_question_original_df.to_csv(f'{folder_name}/lesson_by_q.csv', index=False, encoding='utf-8-sig')
median_df.to_csv(f'{folder_name}/median_data.csv', index=False, encoding='utf-8-sig')
content_question_df.to_csv(f'{folder_name}/content_q.csv', index=False, encoding='utf-8-sig')
revenue_funnel_df.to_csv(f'{folder_name}/revenue_funnel.csv', index=False, encoding='utf-8-sig')
question_payment_df.to_csv(f'{folder_name}/question_payment.csv', index=False, encoding='utf-8-sig')
clicker_vs_non_df.to_csv(f'{folder_name}/clicker_vs_non.csv', index=False, encoding='utf-8-sig')
question_user_df.to_csv(f'{folder_name}/question_user.csv', index=False, encoding='utf-8-sig')
revenue_summary.to_csv(f'{folder_name}/revenue_summary.csv', index=False, encoding='utf-8-sig')
arppu_result.to_csv(f'{folder_name}/arppu_result.csv', index=False, encoding='utf-8-sig')
revenue_contribution_df.to_csv(f'{folder_name}/rev_contribution.csv', index=False, encoding='utf-8-sig')

print(f"✅ CSV 파일 내보내기 완료! '{folder_name}' 폴더 확인.")